# Sistema di Visione 3D

# 04 Acquisizione Oggetto

In [1]:
import os
import cv2 as cv
import numpy as np
import pickle
import pandas as pd
import plotly.offline as pyo
import plotly.graph_objs as go
import plotly.offline as pyo
import open3d as o3d
from os import listdir, path
from matplotlib import pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.cluster import DBSCAN

# Directory e costanti
IMGDIR      = 'images'
OGGETTO     = os.path.join(IMGDIR,'oggetto/cropped')
OUTDIR      = 'outputs'
PROFILO     = os.path.join(IMGDIR,'profilo')
Z_BASE      = 50 # altezza della base in mm

FILE_NAME = '30o34_CROPPED.jpg'

In [ ]:
with open(os.path.join(OUTDIR, 'cameraMatrix.dump'), 'rb') as f:
    cameraMatrix = pickle.load(f)
    print("Matrice della telecamera caricata:")
    print(cameraMatrix)

with open(os.path.join(OUTDIR, 'distCoeffs.dump'), 'rb') as f:
    distCoeffs = pickle.load(f)
    print("Coefficienti di distorsione caricati:")
    print(distCoeffs)

with open(os.path.join(OUTDIR, 'newCameraMatrix.dump'), 'rb') as f:
    newcameramtx = pickle.load(f)
    print("Nuova matrice della telecamera caricata:")
    print(newcameramtx)

with open(os.path.join(OUTDIR, 'roi.dump'), 'rb') as f:
    roi = pickle.load(f)
    print("ROI caricata:")
    print(roi)

In [3]:
def GetRedLine(imgName):
    # Carica l'immagine
    img = cv.imread(imgName, cv.IMREAD_COLOR)
    assert img is not None, "ERROR: File NOT FOUND!"
    
    # Converti l'immagine in scala di grigi
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    
    # Estrai il canale rosso
    imgR = img[:, :, 2]

    # Rimuovi la distorsione dall'immagine
    undistorted = cv.undistort(gray, cameraMatrix, distCoeffs, None, newcameramtx)
    undistortedR = cv.undistort(imgR, cameraMatrix, distCoeffs, None, newcameramtx)

    # Ritaglia l'immagine
    x, y, w, h = roi
    cropped = undistorted[y:y+h, x:x+w]
    croppedR = undistortedR[y:y+h, x:x+w]
    
    # Visualizza l'immagine ritagliata del canale rosso
    plt.imshow(croppedR, cmap='gray')
    plt.title("Immagine ritagliata del canale rosso")
    plt.show()

    gray = croppedR.copy()
    h, w = gray.shape

    # Applica un filtro Gaussiano per ridurre il rumore
    blurred = cv.GaussianBlur(croppedR, (5, 5), 0)

    # Applica una soglia per ottenere una maschera binaria
    x_min = 800
    ret, th2 = cv.threshold(blurred[:, x_min:900], 230 , 255, cv.THRESH_BINARY)
    restored_th2 = np.zeros_like(croppedR)  # Crea un array vuoto con la stessa forma di 'cropped'
    restored_th2[:, x_min:x_min+th2.shape[1]] = th2

    # Applica operazioni morfologiche per pulire l'immagine binaria
    kernel = np.ones((3, 3), np.uint8)
    morphed = cv.morphologyEx(restored_th2, cv.MORPH_OPEN, kernel)
    morphed = cv.morphologyEx(morphed, cv.MORPH_CLOSE, kernel)

    # Visualizza l'immagine binaria pulita
    fig, ax = plt.subplots()
    ax.imshow(morphed, cmap='gray')
    plt.title("Immagine binaria pulita")
    plt.tight_layout()
    plt.show()

    # Inizializza un array per memorizzare le coordinate della linea rossa
    linegray = np.zeros((h, 2), dtype=float)
    for j in range(h):
        nzero = np.where(morphed[j, :] != 0)
        if len(nzero[0]) > 0:
            avg = np.sum(gray[j, nzero] * nzero) / np.sum(gray[j, nzero])
            linegray[j] = np.array([avg, j])
        else:
            linegray[j] = np.array([np.nan, j])  # Gestisci i valori NaN

    return linegray

In [4]:
def cluster_laser_points(points, eps=5.0, min_samples=10):
    """
    Esegue la clusterizzazione DBSCAN sui punti laser
    per eliminare il rumore e isolare i cluster reali corrispondenti alla lama.
    
    Parametri:
    - points: array NumPy di forma (N, 2) o (N, 3)
    - eps: distanza massima tra due punti per essere considerati nel medesimo cluster
    - min_samples: numero minimo di campioni richiesti per formare un cluster
    
    Ritorna:
    - labels: array con le etichette di cluster assegnate a ogni punto
              (-1 indica i punti considerati rumore)
    - clusters: lista di array, ciascuno contenente i punti di un cluster
    """
    # Esegui la clusterizzazione DBSCAN
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(points)
    labels = db.labels_
    unique_labels = set(labels)
    
    # Raggruppa i punti in base alle etichette (tranne -1 che è rumore)
    clusters = []
    for c in unique_labels:
        if c != -1:
            cluster_points = points[labels == c]
            clusters.append(cluster_points)
    
    # Stampa i risultati della clusterizzazione
    print(f"Numero di cluster trovati: {len(clusters)}")
    print(f"Numero di punti di rumore: {np.sum(labels == -1)}")
    
    return labels, clusters

Elaborazione di tutte le immagini per ottenere un unico vettore 

In [ ]:
# Nome del file da elaborare
print(f"Elaborazione file: {FILE_NAME}")

# Costruisci il percorso completo del file
fName = os.path.join(OGGETTO, FILE_NAME)

# Verifica se il file esiste
if os.path.isfile(fName):
    # Calcola la coordinata z basata sul nome del file
    z = int(FILE_NAME[:2]) * 10 + Z_BASE
    
    # Ottieni la linea dall'immagine
    lines = GetRedLine(fName)

# Rimuovi o imputa i valori NaN
imputer = SimpleImputer(strategy='mean')
lines = imputer.fit_transform(lines)

# Esegui la clusterizzazione DBSCAN sui punti laser
labels, clusters = cluster_laser_points(lines, eps=3.0, min_samples=5)

# Visualizza la distribuzione dei cluster (solo per 2D)
plt.figure(figsize=(8, 6))
for idx, cluster_points in enumerate(clusters):
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], label=f"Cluster {idx}")

# Evidenzia i punti di rumore (label = -1)
noise_points = lines[labels == -1]
if len(noise_points) > 0:
    plt.scatter(noise_points[:, 0], noise_points[:, 1], c='k', marker='x', label='Noise')

plt.title("DBSCAN Clustering della lama di luce")
plt.xlabel("X (pixel)")
plt.ylabel("Y (pixel)")
plt.legend()
plt.show()

In [ ]:
display( pd.DataFrame( lines ) ) # i, j

In [ ]:
# Carica i coefficienti della matrice m e i coefficienti della lama di luce
m_file_path = os.path.join(OUTDIR, 'm.dump')
with open(m_file_path, 'rb') as f:
    m = pickle.load(f)
    print("Coefficienti della matrice m caricati:")
    print(m)

with open(os.path.join(OUTDIR, 'lightPlane.dump'), 'rb' ) as f:
    lightPlane = pickle.load( f )
    print("Coefficienti della lama di luce caricati:")
    print(lightPlane)

In [ ]:
# Inizializza un array vuoto per memorizzare i punti calcolati
pts = np.zeros((0, 3), dtype=float)

# Itera su tutti i punti delle linee
for pt in lines:
    # Calcola i coefficienti della matrice A
    a11 = m[0] - m[8] * pt[0]
    a12 = m[1] - m[9] * pt[0]
    a13 = m[2] - m[10] * pt[0]
    a21 = m[4] - m[8] * pt[1]
    a22 = m[5] - m[9] * pt[1]
    a23 = m[6] - m[10] * pt[1]

    # Costruisci la matrice A
    A = np.array([
        [a11, a12, a13],
        [a21, a22, a23],
        [lightPlane[0], lightPlane[1], -1]
    ])

    # Calcola i termini noti del sistema lineare
    b1 = pt[0] - m[3]
    b2 = pt[1] - m[7]
    b = np.array([b1, b2, -lightPlane[2]])

    # Risolvi il sistema lineare A * x = b
    x = np.linalg.solve(A, b)

    # Aggiungi il punto calcolato all'array pts
    pts = np.vstack([pts, (x[0], x[1], x[2])])

pts[:, 2] -= z
pts[:, 2] = -pts[:, 2]

# Visualizza i punti calcolati (xsg, ysg, zsg di ogni punto)
print("Punti calcolati (xsg, ysg, zsg):")
display( pd.DataFrame( pts ) )

In [ ]:
# Inizializza la modalità notebook per Plotly
pyo.init_notebook_mode()

# Crea il grafico 3D per i punti dell'oggetto
tObj = go.Scatter3d(
    x=pts[:, 0], 
    y=pts[:, 1], 
    z=pts[:, 2], 
    mode='markers', 
    marker={'size': 2, 'opacity': 0.8},
    name='Object Points'
)

# Definisci il layout del grafico
layout = go.Layout(
    margin={'l': 0, 'r': 0, 'b': 0, 't': 0},
    scene=dict(
        xaxis=dict(title='X (mm)'),
        yaxis=dict(title='Y (mm)'),
        zaxis=dict(title='Z (mm)')
    )
)

# Crea la figura con i dati e il layout
fig = go.Figure(data=[tObj], layout=layout)

# Visualizza il grafico
pyo.iplot(fig)

In [ ]:
# Crea una nuvola di punti Open3D
pcd = o3d.geometry.PointCloud()

# Aggiungi i punti alla nuvola di punti
pcd.points = o3d.utility.Vector3dVector(pts)

# Salva la nuvola di punti in un file PLY
o3d.io.write_point_cloud(path.join(OUTDIR,'object_points.ply'), pcd)

print("Nuvola di punti salvata con successo.")

In [ ]:
PLY_FILE = path.join(OUTDIR, 'object_points.ply')

# Parametri per analisi e visualizzazione
VOXEL_SIZE = 0.02 # dimensione del voxel per il campionamento
OUTLIER_NB_NEIGHBORS = 20 # numero di vicini per il filtraggio degli outlier
OUTLIER_STD_RATIO = 2.0 # rapporto standard per il filtraggio degli outlier

# Funzione per visualizzare nuvole di punti con Plotly
def plot_point_cloud(points, title, color='blue', size=2, opacity=0.8):
    scatter = go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='markers',
        marker=dict(size=size, opacity=opacity, color=color),
        name=title
    )
    layout = go.Layout(
        title=title,
        margin=dict(l=0, r=0, b=0, t=0),
        scene=dict(
            xaxis=dict(title='X (mm)'),
            yaxis=dict(title='Y (mm)'),
            zaxis=dict(title='Z (mm)')
        )
    )
    fig = go.Figure(data=[scatter], layout=layout)
    pyo.iplot(fig)

# Caricamento del file
if not path.exists(PLY_FILE):
    raise FileNotFoundError(f"Il file {PLY_FILE} non esiste.")
pcd = o3d.io.read_point_cloud(PLY_FILE)
points = np.asarray(pcd.points)

# Filtra i punti per rimuovere valori del piano
points_filtered = points[(points[:, 1] >= -80 ) & (points[:, 1] <= 100)]
pcd.points = o3d.utility.Vector3dVector(points_filtered)

# Visualizza nuvola di punti originale
pyo.init_notebook_mode()
print("Nuvola di Punti Originale")
plot_point_cloud(points, "Nuvola di Punti Originale")

# Filtraggio e decimazione
pcd_down = pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)
pcd_down, _ = pcd_down.remove_statistical_outlier(
    nb_neighbors=OUTLIER_NB_NEIGHBORS, std_ratio=OUTLIER_STD_RATIO
)
points_down = np.asarray(pcd_down.points)
# points_down = points
print("Nuvola di Punti Filtrata")
plot_point_cloud(points_down, "Nuvola di Punti Filtrata")

# Trasla i punti
z_min = np.min(points_down[:, 2])
points_translated = points_down.copy()
points_translated[:, 2] -= z_min

# Visualizza la nuvola di punti traslata come immagine 2D di y e z
plt.figure(figsize=(10, 8))
plt.scatter(points_translated[:, 1], points_translated[:, 2], s=1, c='red', alpha=0.6)
plt.title("Nuvola 2D di Punti Oggetto")
plt.xlabel("Y (mm)")
plt.ylabel("Z (mm)")
plt.grid(True)
plt.show()

# Ordina i punti per la coordinata y
points_translated = points_translated[np.argsort(points_translated[:, 1])]

# Estrai l'estensione del file da FILE_NAME
file_base, file_ext = path.splitext(FILE_NAME)
output_file_name = f"{file_base}_profilo{file_ext}"

# Visualizza il profilo dell'oggetto
plt.figure(figsize=(10, 8))
plt.plot(points_translated[:, 1], points_translated[:, 2], 'o ', markersize=2, color='red', alpha=0.6)
plt.title("Profilo Oggetto 6: " + file_base)
plt.xlabel("Y (mm)")
plt.ylabel("Z (mm)")
plt.grid(True)
plt.xlim(-50, 75)
# Imposta dimensione griglia

# Salva l'immagine
plt.savefig(os.path.join(PROFILO, output_file_name))

plt.show()